In [1]:
# ─────────────────────────────────────────
# 기본 모듈
# ─────────────────────────────────────────
import os
import operator  # 상태 업데이트 방식 정의할 때 사용
from typing import TypedDict, Annotated, Sequence
# TypedDict  : 딕셔너리의 키/값 타입을 명시하는 클래스
# Annotated  : 타입에 추가 정보를 붙일 때 사용
# Sequence   : 리스트/튜플 등 순서 있는 컬렉션 타입
from dotenv import load_dotenv
load_dotenv(override=True)

# ─────────────────────────────────────────
# LangChain / LangGraph 관련 import
# ─────────────────────────────────────────
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
# BaseMessage  : 모든 메시지의 부모 클래스
# HumanMessage : 사용자 메시지
# AIMessage    : LLM 응답 메시지
# ToolMessage  : Tool 실행 결과 메시지 (3번의 role:user Observation과 같은 역할)

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
# StateGraph : 노드와 엣지로 워크플로우를 정의하는 클래스
# END        : 그래프 종료를 나타내는 특수 상수

from langgraph.prebuilt import ToolNode
# ToolNode : Tool 실행을 자동으로 처리하는 사전 빌드 노드
#            3번의 _execute_tool() 을 자동으로 대체


# ─────────────────────────────────────────
# AgentState : 그래프 전체에서 공유되는 상태 객체
#
# TypedDict  : 딕셔너리인데 키/값 타입이 정해져 있음
# messages   : 대화 이력을 담는 리스트
#
# Annotated[Sequence[BaseMessage], operator.add]
# → 이 상태가 업데이트될 때 기존 리스트에 새 메시지를 추가(add)
# → operator.add = 리스트끼리 더하기 (append가 아닌 +)
# → 덮어쓰지 않고 누적되는 것이 핵심
# ─────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]


# ─────────────────────────────────────────
# Tool 정의 (5번과 동일하게 @tool 데코레이터 사용)
# ─────────────────────────────────────────
@tool
def get_current_time(location: str) -> str:
    '''한국을 기준으로 입력한 도시의 현재 현지 시간을 반환합니다. 도시명은 한글로 입력하세요'''
    from datetime import datetime
    try:
        from zoneinfo import ZoneInfo
    except ImportError:
        return '현재 환경에서 zoneinfo를 사용할 수 없습니다.'

    timezone_map = {
        '서울': 'Asia/Seoul', '뉴욕': 'America/New_York',
        '런던': 'Europe/London', '도쿄': 'Asia/Tokyo',
        '시드니': 'Australia/Sydney', '파리': 'Europe/Paris'
    }

    tz_name = timezone_map.get(location.strip())
    if tz_name is None:
        return f"'{location}' 도시의 타임존을 찾을 수 없습니다."

    seoul_time = datetime.now(ZoneInfo('Asia/Seoul'))
    city_time  = datetime.now(ZoneInfo(tz_name))
    return (
        f"한국 시간({seoul_time.strftime('%Y-%m-%d %H:%M:%S')}) 기준으로 "
        f"{location}의 현재 시간은 {city_time.strftime('%Y-%m-%d %H:%M:%S')} 입니다."
    )

tools = [get_current_time]

# ─────────────────────────────────────────
# ToolNode : tools 리스트를 받아서
# LLM이 tool_calls를 요청하면 자동으로 실행
# 3번의 _execute_tool() 전체를 한 줄로 대체
# ─────────────────────────────────────────
tool_node = ToolNode(tools)

print('상태 정의 및 도구 초기화 완료')

c:\Users\Playdata\miniconda3\envs\llm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


상태 정의 및 도구 초기화 완료


In [2]:
# ─────────────────────────────────────────
# LLM 초기화 + Tool 바인딩
#
# bind_tools() : LLM에게 사용 가능한 Tool 목록을 알려줌
# 5번의 create_agent(llm, tools) 와 비슷하지만
# LangGraph에서는 LLM과 Tool을 분리해서 관리
# → 더 세밀한 제어가 가능
# ─────────────────────────────────────────
model = ChatOpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-4o-mini',
    temperature=0
)
# bind_tools() : 이 LLM은 tools를 사용할 수 있다고 선언
# 내부적으로 Function Calling 방식으로 동작
# → LLM이 tool_calls 필드에 JSON으로 호출 정보를 담아 반환
model_with_tools = model.bind_tools(tools)


# ─────────────────────────────────────────
# call_model : 에이전트 노드 함수
#
# 노드(Node) = 그래프에서 실제 작업을 수행하는 함수
# 규칙 : 반드시 state를 받아서 state 업데이트를 반환
#
# state['messages'] : 지금까지 쌓인 전체 대화 이력
# model_with_tools.invoke() : LLM 호출
# return {'messages': [response]} :
#   → 새 메시지를 리스트로 반환
#   → AgentState의 operator.add 덕분에
#     기존 messages에 자동으로 추가됨 (덮어쓰기 X)
# ─────────────────────────────────────────
def call_model(state: AgentState):
    messages = state['messages']          # 현재까지의 대화 이력
    response = model_with_tools.invoke(messages)  # LLM 호출
    return {'messages': [response]}       # 새 메시지만 반환 → 자동으로 누적


# ─────────────────────────────────────────
# should_continue : 라우팅 함수 (조건부 엣지)
#
# 엣지(Edge) = 노드와 노드를 연결하는 흐름
# 조건부 엣지 = 상황에 따라 다른 노드로 이동
#
# LLM 응답의 마지막 메시지를 확인해서
# tool_calls 있음 → 'continue' → action 노드로 이동
# tool_calls 없음 → 'end'      → 그래프 종료
#
# 3번에서 직접 짰던 이 로직과 같은 역할 :
# if action_name == 'Finish': return
# elif action_name: execute_tool()
# ─────────────────────────────────────────
def should_continue(state: AgentState):
    messages = state['messages']
    last_message = messages[-1]  # 가장 최근 LLM 응답

    # ─────────────────────────────────────────
    # tool_calls : LLM이 Tool을 호출하고 싶을 때
    #              응답에 포함되는 필드
    # 있으면  → 아직 Tool 실행이 필요함 → 계속 진행
    # 없으면  → LLM이 최종 답변을 냈음  → 종료
    # ─────────────────────────────────────────
    if last_message.tool_calls:
        return 'continue'  # → action 노드로
    return 'end'            # → END로


print('에이전트 노드와 라우터 준비')

에이전트 노드와 라우터 준비


In [3]:
# ─────────────────────────────────────────
# StateGraph : 노드와 엣지로 워크플로우를 정의하는 클래스
# AgentState를 인자로 넘겨서
# 이 그래프가 어떤 상태를 공유할지 선언
# ─────────────────────────────────────────
workflow = StateGraph(AgentState)


# ─────────────────────────────────────────
# 노드 등록
# add_node(이름, 함수)
# 이름  : 나중에 엣지 연결할 때 쓰는 식별자
# 함수  : 실제 실행될 노드 함수
#
# 'agent'  → call_model()   : LLM 호출 노드
# 'action' → tool_node      : Tool 실행 노드 (ToolNode)
# ─────────────────────────────────────────
workflow.add_node('agent', call_model)   # 추론 노드
workflow.add_node('action', tool_node)   # 행동(Tool 실행) 노드


# ─────────────────────────────────────────
# 시작점 설정
# 그래프가 실행되면 가장 먼저 'agent' 노드로 진입
# ─────────────────────────────────────────
workflow.set_entry_point('agent')


# ─────────────────────────────────────────
# 조건부 엣지 등록
# add_conditional_edges(현재노드, 라우터함수, 분기맵)
#
# 현재노드  : 'agent' 노드가 끝난 후 분기
# 라우터    : should_continue() 의 반환값으로 판단
# 분기맵    : 반환값 → 다음 노드 매핑
#
# should_continue() 가 'continue' 반환 → 'action' 노드로
# should_continue() 가 'end' 반환      → END (그래프 종료)
#
# 3번에서 직접 짰던 이 로직을 대체 :
# if action_name == 'Finish': return action_input
# elif action_name: observation = execute_tool()
# ─────────────────────────────────────────
workflow.add_conditional_edges(
    'agent',          # 어떤 노드의 출력을 보고 분기할지
    should_continue,  # 분기 판단 함수
    {
        'continue': 'action',  # 'continue' 반환 → action 노드
        'end': END             # 'end' 반환      → 종료
    }
)


# ─────────────────────────────────────────
# 일반 엣지 등록
# add_edge(시작노드, 끝노드)
# action 노드가 끝나면 무조건 agent 노드로 돌아감
#
# 이게 바로 ReAct 루프를 만드는 핵심 !
# agent → (tool 필요) → action → agent → (tool 필요) → action ...
# agent → (tool 불필요) → END
# ─────────────────────────────────────────
workflow.add_edge('action', 'agent')


# ─────────────────────────────────────────
# 그래프 컴파일
# compile() : 정의한 노드와 엣지를 실행 가능한 앱으로 변환
# 이후 app.invoke() 또는 app.stream() 으로 실행
# ─────────────────────────────────────────
app = workflow.compile()

print('ReAct 워크플로우 컴파일 완료')

ReAct 워크플로우 컴파일 완료


In [4]:
# ─────────────────────────────────────────
# HumanMessage : 사용자 메시지를 LangChain 메시지 객체로 생성
# 3번에서 {'role':'user', 'content':'...'} 로 직접 만들었던 것을
# HumanMessage(content='...') 로 대체
# ─────────────────────────────────────────
inputs = {
    'messages': [HumanMessage(content='시드니의 현재 시간은 언제인가요?')]
}

print('== LangGraph ReAct 실행 흐름 ==')

# ─────────────────────────────────────────
# app.stream() : 그래프를 실행하면서
# 각 노드가 완료될 때마다 결과를 순서대로 반환
# → 3번의 for step in range(max_steps) 루프를 대체
#
# app.invoke() 와의 차이 :
# invoke() → 전체 실행 후 최종 결과만 반환
# stream() → 노드 하나씩 완료될 때마다 중간 결과 반환
#             실행 흐름을 단계별로 추적할 수 있음
# ─────────────────────────────────────────
for output in app.stream(inputs):

    # ─────────────────────────────────────────
    # output : 딕셔너리
    # key    : 방금 완료된 노드 이름 ('agent' 또는 'action')
    # value  : 그 노드가 반환한 상태 업데이트
    # ─────────────────────────────────────────
    for node_name, state_update in output.items():
        print(f'\n[노드 실행 완료]: {node_name}')

        # 해당 노드에서 마지막으로 추가된 메시지
        latest_message = state_update['messages'][-1]

        # ─────────────────────────────────────────
        # isinstance() : 객체가 특정 클래스의 인스턴스인지 확인
        # AIMessage    : LLM이 생성한 메시지
        # ToolMessage  : Tool 실행 결과 메시지
        # ─────────────────────────────────────────
        if isinstance(latest_message, AIMessage):

            # ─────────────────────────────────────────
            # tool_calls 있음 → LLM이 Tool 호출을 요청한 상태
            # tool_calls[0]   → 첫번째 Tool 호출 정보
            # ['name']        → 호출할 Tool 이름
            # ─────────────────────────────────────────
            if hasattr(latest_message, 'tool_calls') and latest_message.tool_calls:
                tool_call = latest_message.tool_calls[0]
                print(f" → Thought: 도구 '{tool_call['name']}' 호출 결정")

            # tool_calls 없음 → 최종 답변 완성
            else:
                print(f' → Final Answer: {latest_message.content}')

        # ─────────────────────────────────────────
        # ToolMessage : Tool 실행 결과
        # 3번에서 obs_msg = f'Observation: {observation}' 로
        # 직접 만들었던 것을 LangGraph가 자동으로 생성
        # ─────────────────────────────────────────
        elif isinstance(latest_message, ToolMessage):
            print(f' → Observation: {latest_message.content}')

print('\n=== 실행 종료 ===')

== LangGraph ReAct 실행 흐름 ==

[노드 실행 완료]: agent
 → Thought: 도구 'get_current_time' 호출 결정

[노드 실행 완료]: action
 → Observation: 한국 시간(2026-06-01 16:25:23) 기준으로 시드니의 현재 시간은 2026-06-01 17:25:23 입니다.

[노드 실행 완료]: agent
 → Final Answer: 현재 시드니의 시간은 2026년 6월 1일 17시 25분 23초입니다.

=== 실행 종료 ===
